# Session 22 — Research and Apply Data Cleaning with Pandas

## Part 1: Research

### 1. What is data cleaning, and why is it important?
Data cleaning is the process of finding and fixing or removing inaccurate, incomplete, duplicated, or incorrectly formatted data. It is important because poor-quality data can lead to misleading analysis and incorrect decisions.

### 2. How do `isnull()` and `isna()` identify missing values?
Both methods return a Boolean result for each value: `True` when the value is missing and `False` otherwise. In Pandas, `isnull()` and `isna()` are equivalent aliases.

### 3. What is the difference between `dropna()` and `fillna()`?
`dropna()` removes rows or columns containing missing values, depending on its settings. `fillna()` replaces missing values with a chosen value or a value calculated from the data. The best choice depends on how much data is missing and what the column represents.

### 4. How does `drop_duplicates()` work?
`drop_duplicates()` removes repeated rows. By default, it keeps the first occurrence and removes later rows that match it; parameters can change which occurrence is kept or which columns are compared.

### 5. How can Pandas convert columns to appropriate data types using `astype()` and `to_datetime()`?
`astype()` converts a column to a specified data type, such as integer or string. `to_datetime()` converts date-like values into Pandas datetime values, which makes date filtering and date calculations easier. For messy values, `errors='coerce'` can turn invalid values into missing values rather than stopping execution.

## Part 2: Practical Application

The example dataset has 24 records. It intentionally includes missing values, duplicate records, numeric values stored as text, and invalid salary strings. The original dataset is also included as `employees_original.csv`.


### 1. Load the original dataset

In [ ]:
import pandas as pd
from pathlib import Path

# If running from the same folder as the notebook:
df_original = pd.read_csv("employees_original.csv")
df = df_original.copy()

df.head()


### 2. Identify and count missing values

In [ ]:
# Empty CSV fields are loaded as missing values (NaN).
print("Missing values by column:")
print(df.isnull().sum())

# isna() is equivalent to isnull().
print("\nTotal missing cells:", df.isna().sum().sum())


### 3. Detect duplicate records

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)
df[df.duplicated(keep=False)]


### 4. Clean the data

#### Cleaning decisions
- Convert empty strings to missing values.
- Remove exact duplicate rows, keeping the first occurrence.
- Convert `Age` and `Salary` to numeric values; invalid values become missing.
- Fill missing ages with the median age, because age is numeric and the median is less affected by extreme values.
- Fill missing salaries with the median salary for the employee's department; use the overall median only if a department has no valid salary values.
- Fill missing email addresses with `Unknown` because no email can be safely inferred.
- Convert `Join_Date` to a datetime type.


In [ ]:
# Normalize blank text cells as missing values.
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Remove exact duplicate records.
df = df.drop_duplicates().copy()

# Convert numeric columns. Invalid strings become NaN.
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Fill missing ages with the median.
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill missing salaries with the median salary in each department.
department_medians = df.groupby('Department')['Salary'].transform('median')
df['Salary'] = df['Salary'].fillna(department_medians)
df['Salary'] = df['Salary'].fillna(df['Salary'].median())

# Email cannot be reconstructed from the other fields.
df['Email'] = df['Email'].fillna('Unknown')

# Convert join dates to datetime.
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce')

# Use an integer type for age after filling missing values.
df['Age'] = df['Age'].round().astype('int64')

df.head()


### 5. Inspect the cleaned dataset and data types

In [ ]:
print("Cleaned shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values after cleaning:")
print(df.isna().sum())

print("\nDuplicate rows after cleaning:", df.duplicated().sum())


### 6. Compare the dataset before and after cleaning

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Missing cells', 'Duplicate rows'],
    'Before cleaning': [
        len(df_original),
        df_original.shape[1],
        df_original.isna().sum().sum(),
        df_original.duplicated().sum()
    ],
    'After cleaning': [
        len(df),
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum()
    ]
})
comparison


### 7. Save the original and cleaned datasets

In [ ]:
df_original.to_csv("employees_original.csv", index=False)
df.to_csv("employees_cleaned.csv", index=False)

print("Saved: employees_original.csv")
print("Saved: employees_cleaned.csv")


## Short conclusion
The cleaning process removed exact duplicates, converted text-based numeric columns and dates to appropriate types, and handled missing or invalid values using explicit rules. The comparison table helps verify the change in row count, missing values, and duplicate records.
